# Figures 5 and 6 - Potential of mean force, Yukawa + Lennard-Jones

Two-body Monte Carlo simulations with **Faunus** of two rigid antibodies at
amino-acid resolution, now with the full effective interaction: a screened
Coulomb (Yukawa) term plus a Lennard-Jones short-range attraction. The PMF is
obtained from g(r) as `U(r)/kT = -ln[g(r)/g_bulk]`.

The screening length is set per ionic strength through `kappa()`. Structures
`mAb-1` at 7 and 57 mM are run, each in its own sub-directory, reading from the
shared `../pqr_inputs/` folder.

**Requires the hard-sphere data first:** the plot cells overlay the hard-sphere
PMF from the `Fig_2` notebook (`../Fig_2/<name>/rdf.dat`). Run `Fig_2` before
plotting here, and keep that folder named `Fig_2`.

**How to run:** install the environment, then run top to bottom. Edit
`run_faunus()` to submit to a cluster instead.

## Import packages

In [ ]:
import os
import subprocess
import numpy as np
from jinja2 import Template
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 16, "legend.frameon": True, "figure.figsize": [5, 5],
                     "xtick.major.size": 7, "ytick.major.size": 7, "legend.labelspacing": 1})


## Functions

In [ ]:
def create_input(workdir, structure, macro, micro, nstep, nskip, Dl, eps):
    """Render the Faunus input (input.yml) for one run from the template below.
    Dl is the Debye length (angstrom), eps the Lennard-Jones well depth (kJ/mol)."""
    with open("input.yml", "w") as f:
        return f.write(Input.render(workdir=workdir, structure=structure,
                                    macro=macro, micro=micro, nstep=nstep,
                                    nskip=nskip, Dl=Dl, eps=eps))


def run_faunus(use_state=False):
    """Run Faunus locally in the current run directory.

    Requires `faunus` and `yason.py` on PATH (installed by the conda
    environment, see ../environment.yml). Output goes to `out`, errors to `err`.
    To run on a cluster instead, replace the body of this function with your
    own submission command (e.g. write an sbatch script and call `sbatch`)."""
    state = "--state state.json " if use_state else ""
    cmd = ("set -o pipefail; export OMP_NUM_THREADS=1; "
           f"yason.py input.yml | faunus {state}-v 6 > out 2> err")
    subprocess.run(["bash", "-c", cmd], check=True)


def kappa(conc, Z, rhob, rhos, sigma):
    """Inverse Debye length (1/nm) used to set the Yukawa screening length.
    conc is 0 in these runs, so the screening depends on the salt term rhob
    (ionic strength in mM) only; Z and sigma then have no effect."""
    lB = 0.714               # Bjerrum length, nm
    Na = 6.02214076e23       # 1/mol
    rho = 4.068e15           # particles per ml
    phi = conc * rho * 1e-21 * np.pi * sigma ** 3 / 6
    PHI = 1 / (1 - phi)
    k = 4 * np.pi * lB * (PHI * conc * Z * rho * 1e-21
                          + 2 * rhob * Na * 1e-27 + 2 * rhos * Na * 1e-27)
    return np.sqrt(k)


# PMF Yukawa + Lennard-Jones potential - runs

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------

Input = Template("""
temperature: 298.15
geometry: {type: cuboid, length: [300, 300, 2000] }
mcloop: {micro: {{micro}}, macro: {{macro}}}
random: {seed: hardware}
energy:
    - nonbonded_coulomblj:
        cutoff_g2g: {{ 180 + 8*Dl }}
        coulomb: { type: yukawa, debyelength: {{Dl}}, epsr: 78.7, shift: True, cutoff: {{8*Dl}} }
        lennardjones: { mixing: LB}
                
atomlist:
    - CTR:  {mw: 16.0,  sigma: 3.13, eps: {{eps}}, q: -1 }
    - HCTR: {mw: 16.0,  sigma: 3.13, eps: {{eps}}, q: 0 }
    - NTR:  {mw: 14.0,  sigma: 2.99, eps: {{eps}}, q: 0 }
    - HNTR: {mw: 14.0,  sigma: 2.99, eps: {{eps}}, q: 1 }
    - ALA:  {mw: 66.0,  sigma: 5.02, eps: {{eps}}, q: 0 }
    - ILE:  {mw: 102.0, sigma: 5.8,  eps: {{eps}}, q: 0 }
    - LEU:  {mw: 102.0, sigma: 5.8,  eps: {{eps}}, q: 0 }
    - MET:  {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 0 }
    - PHE:  {mw: 138.0, sigma: 6.41, eps: {{eps}}, q: 0 }
    - VAL:  {mw: 90.0,  sigma: 5.56, eps: {{eps}}, q: 0 }
    - PRO:  {mw: 90.0,  sigma: 5.56, eps: {{eps}}, q: 0 }
    - TRP:  {mw: 176.0, sigma: 6.95, eps: {{eps}}, q: 0 }
    - GLY:  {mw: 54.0,  sigma: 4.69, eps: {{eps}}, q: 0 }
    - ARG:  {mw: 144.0, sigma: 6.50, eps: {{eps}}, q: 0 }
    - HARG: {mw: 144.0, sigma: 6.50, eps: {{eps}}, q: 1 }
    - LYS:  {mw: 116.0, sigma: 6.05, eps: {{eps}}, q: 0 }
    - HLYS: {mw: 116.0, sigma: 6.05, eps: {{eps}}, q: 1 }
    - ASP:  {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: -1 }
    - HASP: {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: 0 }
    - LASP: {mw: 110.0, sigma: 5.94, eps: {{eps}}, q: 2 }
    - GLU:  {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: -1 }
    - HGLU: {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 0 }
    - LGLU: {mw: 122.0, sigma: 6.16, eps: {{eps}}, q: 2 }
    - GLN:  {mw: 120.0, sigma: 6.12, eps: {{eps}}, q: 0 }
    - ASN:  {mw: 108.0, sigma: 5.91, eps: {{eps}}, q: 0 }
    - HIS:  {mw: 130.0, sigma: 6.29, eps: {{eps}}, q: 0 }
    - HHIS: {mw: 130.0, sigma: 6.29, eps: {{eps}}, q: 1 }
    - SER:  {mw: 82.0,  sigma: 5.39, eps: {{eps}}, q: 0 }
    - THR:  {mw: 94.0,  sigma: 5.64, eps: {{eps}}, q: 0 }
    - TYR:  {mw: 154.0, sigma: 6.65, eps: {{eps}}, q: -1 }
    - HTYR: {mw: 154.0, sigma: 6.65, eps: {{eps}}, q: 0 }
    - CYS:  {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: -1 }
    - HCYS: {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: 0 }
    - CYX:  {mw: 103.0, sigma: 5.72, eps: {{eps}}, q: 0 }                

moleculelist:
    - mAb_G_1:
        structure: {{workdir}}/../pqr_inputs/{{structure}}
        insdir: [0,0,0]
        insoffset: [0,0,0]
        keepcharges: True
        keeppos: False
        rigid: True

    - mAb_G_2:
        structure: {{workdir}}/../pqr_inputs/{{structure}}
        insdir: [0,0,0]
        insoffset: [0,0,150]
        keepcharges: True
        keeppos: False
        rigid: True

insertmolecules:
    - mAb_G_1: {N: 1, inactive: False}
    - mAb_G_2: {N: 1, inactive: False}
moves:    
    - moltransrot: {molecule: mAb_G_1, dir: [0,0,0], dprot: 1, dp: 0}
    - moltransrot: {molecule: mAb_G_2, dir: [0,0,1], dprot: 1, dp: 100}

analysis:
    - sanity: { nstep: {{nstep}} }
    - density: { nstep: {{nstep}} }
    - savestate: { file: state.json }
    - savestate: { file: confout.pqr }
    - xtcfile: { file: traj.xtc, nstep: {{nstep}}, nskip: {{nskip}} }
    - systemenergy: { file: energy.dat, nstep: {{nstep}}, nskip: {{nskip}} }
    - molrdf: {file: rdf.dat, nstep: 1, dr: 1, name1: mAb_G_1, name2: mAb_G_2, dim: 1}
      

""")

# structures to run (read from ../pqr_inputs/)
structures_list = ['mAb-1_7mM.pqr', 'mAb-1_57mM.pqr']

to_kJmol = 2.476                      # kT at 298.15 K in kJ/mol
eps = round(0.075 * to_kJmol, 4)      # Lennard-Jones well depth (kJ/mol)

micro = 1000
macro = 1000
nstep = 1000
nskip = 0

for structure in structures_list:
    name = os.path.splitext(structure)[0]
    os.makedirs(name, exist_ok=True)
    os.chdir(name)

    # net charge z and effective size sigma (nm) used to set the Debye length
    if structure == 'mAb-1_7mM.pqr':
        z, sigma, I = 31.3, 9.64, 7
    elif structure == 'mAb-1_57mM.pqr':
        z, sigma, I = 36.3, 9.0, 57
    Dl = 10 / kappa(0, z, I, 0, sigma)        # Debye length, angstrom

    create_input(workdir, structure, macro, micro, nstep, nskip, Dl, eps)
    run_faunus(use_state=os.path.isfile('state.json'))
    os.chdir(workdir)


# Plot PMF - mAb-1, I = 7 mM

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------

structure = 'mAb-1_7mM.pqr'
name = os.path.splitext(structure)[0]

plt.figure(figsize=(7, 5))

# Yukawa + Lennard-Jones PMF (this notebook)
r, g_r = np.loadtxt(os.path.join(name, 'rdf.dat'), unpack=True, usecols=(0, 1))
plt.plot(r / 10, -np.log(g_r / np.mean(g_r[-100:])),
         '.', markersize=5, color='tab:blue', label='mAb-1, I = 7 mM',
         fillstyle='none', linewidth=0)

# hard-sphere reference from the Fig_2 notebook
r_HS, g_r_HS = np.loadtxt(os.path.join(workdir, '..', 'Fig_2', name, 'rdf.dat'), unpack=True)
plt.plot(r_HS / 10, -np.log(g_r_HS / np.mean(g_r_HS[-100:])), '.k', label='hard sphere')

plt.xlabel('r [nm]')
plt.ylabel(r'$U(r)/k_BT$')
plt.xlim(0, 30)
plt.ylim(0, 10)
plt.tight_layout()
plt.show()


# Plot PMF - mAb-1, I = 57 mM

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------

structure = 'mAb-1_57mM.pqr'
name = os.path.splitext(structure)[0]

plt.figure(figsize=(7, 5))

# Yukawa + Lennard-Jones PMF (this notebook)
r, g_r = np.loadtxt(os.path.join(name, 'rdf.dat'), unpack=True, usecols=(0, 1))
plt.plot(r / 10, -np.log(g_r / np.mean(g_r[-100:])),
         '.', markersize=5, color='tab:red', label='mAb-1, I = 57 mM',
         fillstyle='none', linewidth=0)

# hard-sphere reference from the Fig_2 notebook
r_HS, g_r_HS = np.loadtxt(os.path.join(workdir, '..', 'Fig_2', name, 'rdf.dat'), unpack=True)
plt.plot(r_HS / 10, -np.log(g_r_HS / np.mean(g_r_HS[-100:])), '.k', label='hard sphere')

plt.xlabel('r [nm]')
plt.ylabel(r'$U(r)/k_BT$')
plt.xlim(0, 20)
plt.ylim(-2, 10)
plt.tight_layout()
plt.show()
